In [1]:
import pandas as pd
import numpy as np
import json
import glob
import csv
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import spacy
from nltk.corpus import stopwords
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models

/var/folders/d9/sbfhfygx5vx9mlfxxhcxrb280000gn/T/ipykernel_39990/2475999428.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
def load_data(file):
    with open (file, "r", encoding="utf-8") as f:
        data = json.load(f)
    return (data)

def write_data(file, data):
    with open (file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [3]:
stopwords = stopwords.words("english")
stopwords.append("be")
print(stopwords)

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [4]:
# Load the JSON data from the file
with open("kenny_transcripts.json", "r") as file:
    data = json.load(file)

# Get the keys of the JSON object
fields = data.keys()

# Print the fields
print(fields)


dict_keys(['Transcripts'])


In [8]:
data = load_data("kenny_transcripts.json")["Transcripts"]
print (data[0][0:90])
# print (data[1][0:90])

Oh my God. Thank you so much Mumbai. Thank you. Thank you. Really. How are you guys doing?


In [9]:
def lemmatization(texts, allowed_postages=["NOUN", "ADJ", "VERB", "ADV"]):
    nlp = nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    texts_out = []
    for text in data:
        doc = nlp(text)
        new_text = []
        for token in doc:
            if token.pos_ in allowed_postages:
                new_text.append(token.lemma_)
        final = " ".join(new_text)
        texts_out.append(final)
    return (texts_out)

lemmatized_texts = lemmatization(data)
print (lemmatized_texts[0][0:150])

thank so much thank thank really guy do let come guy ’ guy ’ big deal know none know history ’ big deal take bath twice even possible apparently great


In [10]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download()

showing info https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/index.xml


True

In [11]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'write', 'make', 'take', 'subject', 're', 'edu', 'use', 'be', 'know', 'go', 'think', 'come', 'see', 'guy', 'say', 'even', 'year', 'one', 'would', 'find', 'get'])
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/alisha/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
def gen_words(data):
    final = []
    for text in data:
        new = gensim.utils.simple_preprocess(text, deacc=True)
        final.append(new)
    return (final)

data_words = gen_words(lemmatized_texts)

print (data_words[0][0:20])
# print (data_words[1][0:20])

['thank', 'so', 'much', 'thank', 'thank', 'really', 'guy', 'do', 'let', 'come', 'guy', 'guy', 'big', 'deal', 'know', 'none', 'know', 'history', 'big', 'deal']


In [15]:
data_words = remove_stopwords(data_words)
print(data_words[:1][0][:30])

['thank', 'much', 'thank', 'thank', 'really', 'let', 'big', 'deal', 'none', 'history', 'big', 'deal', 'bath', 'twice', 'possible', 'apparently', 'great', 'deal', 'big', 'occasion', 'turn', 'people', 'cheer', 'die', 'soon', 'yaay', 'yea', 'die', 'soon', 'crazy']


In [16]:
#BIGRAMS AND TRIGRAMS
bigram_phrases = gensim.models.Phrases(data_words, min_count=5, threshold=200)
trigram_phrases = gensim.models.Phrases(bigram_phrases[data_words], threshold=100)

bigram = gensim.models.phrases.Phraser(bigram_phrases)
trigram = gensim.models.phrases.Phraser(trigram_phrases)

def make_bigrams(texts):
    return([bigram[doc] for doc in texts])

def make_trigrams(texts):
    return ([trigram[bigram[doc]] for doc in texts])

data_bigrams = make_bigrams(data_words)
data_bigrams_trigrams = make_trigrams(data_bigrams)

print (data_bigrams_trigrams[0])

['thank', 'much', 'thank', 'thank', 'really', 'let', 'big', 'deal', 'none', 'history', 'big', 'deal', 'bath', 'twice', 'possible', 'apparently', 'great', 'deal', 'big', 'occasion', 'turn', 'people', 'cheer', 'die', 'soon', 'yaay', 'yea', 'die', 'soon', 'crazy', 'believe', 'nice', 'kind', 'difficult', 'dance', 'much', 'dance', 'good', 'care', 'else', 'good', 'way', 'dance', 'dance', 'watch', 'applie', 'woman', 'great', 'dancing', 'man', 'stop', 'man', 'stand', 'dance', 'good', 'enough', 'bar', 'bro', 'music', 'dance', 'bro', 'dance', 'dance', 'painful', 'watch', 'club', 'great', 'already', 'fun', 'already', 'high', 'right', 'great', 'plan', 'bro', 'spend', 'buck', 'drink', 'club', 'drink', 'way', 'bro', 'want', 'drink', 'high', 'fast', 'good', 'neat', 'neat', 'neat', 'neat', 'awesome', 'preference', 'kind', 'barbaric', 'place', 'wine', 'great', 'chance', 'still', 'high', 'bro', 'shot', 'kind', 'shot', 'shot', 'many', 'shot', 'shot', 'shot', 'stop', 'shout', 'stop', 'shout', 'care', 'sho

In [17]:
# TF-IDF REMOVAL
from gensim.models import TfidfModel

id2word = corpora.Dictionary(data_bigrams_trigrams)

texts = data_bigrams_trigrams

corpus = [id2word.doc2bow(text) for text in texts]
print (corpus[0][0:20])

tfidf = TfidfModel(corpus, id2word=id2word)

low_value = 0.03
words = []
words_missing_in_tfidf = []
for i in range (0, len(corpus)):
    bow = corpus[i]
    low_value_words = [] # reinitialization to be safe, you can skip this
    tfidf_ids = [id for id, value in bow]
    bow_ids = [id for id, value in bow]
    low_value_words = [id for id, value in tfidf[bow] if value < low_value]
    drops = low_value_words+words_missing_in_tfidf
    for item in drops:
        words.append(id2word[item])
    words_missing_in_tfidf = [id for id in bow_ids if id not in tfidf_ids] # the words with tf-idf score 0 will be missing
    
    new_bow = [b for b in bow if b[0] not in low_value_words and b[0] not in words_missing_in_tfidf]
    corpus[i] = new_bow
    

[(0, 2), (1, 3), (2, 2), (3, 2), (4, 1), (5, 1), (6, 1), (7, 1), (8, 2), (9, 3), (10, 4), (11, 1), (12, 1), (13, 5), (14, 1), (15, 10), (16, 1), (17, 6), (18, 1), (19, 1)]


In [18]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                           id2word=id2word,
                                           num_topics=4,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=10,
                                           alpha="auto")

In [19]:
lda_model.print_topics()

[(0,
  '0.002*"bro" + 0.001*"nice" + 0.001*"mom" + 0.001*"let" + 0.001*"put" + 0.001*"thing" + 0.001*"woman" + 0.001*"maid" + 0.001*"song" + 0.001*"good"'),
 (1,
  '0.018*"bro" + 0.012*"nice" + 0.010*"mom" + 0.008*"let" + 0.008*"put" + 0.008*"look" + 0.008*"thing" + 0.008*"woman" + 0.008*"maid" + 0.007*"want"'),
 (2,
  '0.003*"bro" + 0.002*"mom" + 0.002*"thing" + 0.002*"put" + 0.002*"nice" + 0.002*"look" + 0.002*"people" + 0.002*"maid" + 0.002*"let" + 0.002*"woman"'),
 (3,
  '0.002*"bro" + 0.002*"nice" + 0.002*"mom" + 0.002*"let" + 0.001*"put" + 0.001*"look" + 0.001*"thing" + 0.001*"house" + 0.001*"shit" + 0.001*"time"')]

In [21]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word, mds="mmds", R=7)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1     -0.050639 -0.032195       1        1  99.938883
2      0.015511  0.010036       2        1   0.020788
3      0.017329  0.010833       3        1   0.020478
0      0.017799  0.011325       4        1   0.019851, topic_info=      Term       Freq      Total Category  logprob  loglift
98     bro  41.000000  41.000000  Default   7.0000   7.0000
489   nice  26.000000  26.000000  Default   6.0000   6.0000
461    mom  24.000000  24.000000  Default   5.0000   5.0000
571    put  19.000000  19.000000  Default   4.0000   4.0000
409    let  19.000000  19.000000  Default   3.0000   3.0000
..     ...        ...        ...      ...      ...      ...
461    mom   0.000647  24.128563   Topic4  -6.5611  -2.0016
571    put   0.000641  19.499220   Topic4  -6.5703  -1.7978
674   song   0.000626  11.239608   Topic4  -6.5947  -1.2713
799  woman   0.000635  17.690268   Topic4  -6.5807  -1.7109
736  thing   0.000635  18.579849   Topic4  -6.5805  -1.7597

[67 rows x 6 columns], token_table=      Topic      Freq          Term
term                               
65        1  0.870298      barbaric
93        1  0.869079     boyfriend
98        1  0.988061           bro
168       1  0.868968        cosmic
169       1  0.868339          cost
170       1  0.869357         cough
204       1  0.868901    depressing
209       1  0.868657    dictionary
228       1  0.868571          drum
237       1  0.869410         elder
255       1  0.870147  evolutionary
280       1  0.867822        figure
292       1  0.869130       forever
309       1  0.868791        ghetto
316       1  0.867711   grandfather
349       1  0.869977        honour
354       1  1.004816         house
409       1  1.024698           let
423       1  1.021812          look
433       1  1.017734          maid
461       1  0.994672           mom
485       1  0.869105     nervously
489       1  1.004053          nice
526       1  1.010495        people
571       1  0.974398           put
596       1  0.870412   reservation
616       1  0.869792       sadness
617       1  0.869441        salary
621       1  0.869537          scan
643       1  1.000565          shit
674       1  0.978682          song
728       1  0.868450   technically
736       1  1.022613         thing
743       1  0.870129       tonight
752       1  0.870144       trouble
799       1  1.017509         woman, R=7, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 3, 4, 1])